In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder,FunctionTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV
from imblearn.over_sampling import SMOTEN
from sklearn.feature_selection import SelectKBest,chi2
path='../data/English_grammar.csv'
data=pd.read_csv(path,dtype=str)
target="Authoritative_Source_Annotated_Grammatical_Tense_Category_Past_Tense_Present_Tense_Future_Tense"
x=data.drop(target,axis=1)
y=data[target]
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)
print(y_train.value_counts())
smoten=SMOTEN(random_state=4,k_neighbors=6)
x_train,y_train=smoten.fit_resample(x_train,y_train)#type:ignore
print(y_train.value_counts())
text_transformer=Pipeline(steps=[
    ("imputer",SimpleImputer(strategy="constant",fill_value="None")),
    ("flatten",FunctionTransformer(np.ravel)),
    ("vectorize",TfidfVectorizer(ngram_range=(1,2),min_df=0.01,max_df=0.95))
])
lable_transformer=Pipeline(steps=[
    ("imputer",SimpleImputer(strategy='constant',fill_value="unknown")),
    ("encoder",OneHotEncoder(sparse_output=False,handle_unknown='ignore'))
])
pipeline=Pipeline(steps=[
    ('transform',text_transformer),
    ('feature_select',SelectKBest(score_func=chi2)),
    ('train',RandomForestClassifier())
])
params={
    'train__n_estimators':[50,100,200],
    'train__criterion':['gini','entropy','log_loss'],
    'train__max_depth':[None,3,5]
}
cls=GridSearchCV(estimator=pipeline,param_grid=params,scoring="accuracy",n_jobs=-1,cv=2)
cls.fit(x_train,y_train)#type:ignore
print(cls.best_params_)
y_predict=cls.predict(x_test)
print(classification_report(y_test,y_predict))




Authoritative_Source_Annotated_Grammatical_Tense_Category_Past_Tense_Present_Tense_Future_Tense
Future Tense     3875
Present Tense    3696
Past Tense       3081
Name: count, dtype: int64
Authoritative_Source_Annotated_Grammatical_Tense_Category_Past_Tense_Present_Tense_Future_Tense
Future Tense     3875
Present Tense    3875
Past Tense       3875
Name: count, dtype: int64


c:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:782: UserWarning: k=200 is greater than n_features=125. All the features will be returned.
  warnings.warn(


{'train__criterion': 'gini', 'train__max_depth': None, 'train__n_estimators': 50}
               precision    recall  f1-score   support

 Future Tense       0.99      0.98      0.98       969
   Past Tense       0.93      0.92      0.93       770
Present Tense       0.93      0.95      0.94       925

     accuracy                           0.95      2664
    macro avg       0.95      0.95      0.95      2664
 weighted avg       0.95      0.95      0.95      2664

